# Montly GDP with Stock and Watson (2010) Methods

In [16]:
import pandas as pd
from pathlib import Path
import os
import time
import numpy as np
from functools import reduce
import csv
from typing import Union
import akshare as ak

In [17]:
os.chdir('/Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data')
print("Current Directory:", os.getcwd())

Current Directory: /Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data


## Consumption

In [18]:
consump_df = pd.read_csv(
    "Social Consumption Retail Sales.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)
consump_df = consump_df.iloc[:2, :].copy()

consump_df= consump_df.T.rename_axis("date").reset_index()

# 2) clean date text like "2026年2月" -> "2026-02"
consump_df["date"] = pd.to_datetime(consump_df["date"], format="%Y年%m月", errors="coerce")

consump_df.columns = ["date", "current consumption", "cumulative consumption"]

# Feb cumulative by year
year = consump_df["date"].dt.year
month = consump_df["date"].dt.month
feb_cum_by_year = consump_df.loc[month.eq(2)].set_index(year[month.eq(2)])["cumulative consumption"]

# fill Jan and Feb current = (Feb cumulative / 2)
for m in [1, 2]:
    mask = month.eq(m) & consump_df["current consumption"].isna()   # only fill missing
    consump_df.loc[mask, "current consumption"] = year[mask].map(feb_cum_by_year) / 2

## Trade

In [19]:
trade_df = pd.read_csv(
    "Export-Import.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)

trade_df = trade_df.iloc[[12], :].copy()

trade_df = trade_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
trade_df["date"] = pd.to_datetime(trade_df["date"], format="%Y年%m月", errors="coerce")

trade_df.columns = ["date", "trade balance"]

## Fixed Asset Investment

In [20]:
fai_df = pd.read_csv(
    "Fixed Asset Investment.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)
fai_df = fai_df.iloc[[0], :].copy()

fai_df = fai_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
fai_df["date"] = pd.to_datetime(fai_df["date"], format="%Y年%m月", errors="coerce")

fai_df.columns = ["date", "fai growth"]


In [21]:
urbanfai_df = ak.macro_china_gdzctz()

urbanfai_df["月份"] = pd.to_datetime(urbanfai_df["月份"], format="%Y年%m月份", errors="coerce")

years = sorted(
    y for y in urbanfai_df["月份"].dt.year.dropna().astype(int).unique()
    if y >= 2000
)

jan_dates = pd.to_datetime([f"{y}-01-01" for y in years])
missing_jan = jan_dates[~jan_dates.isin(urbanfai_df["月份"].values)]

if len(missing_jan) > 0:
    jan_rows = pd.DataFrame({"月份": missing_jan})
    urbanfai_df = pd.concat([urbanfai_df, jan_rows], ignore_index=True)
    urbanfai_df = urbanfai_df.sort_values("月份").reset_index(drop=True)

urbanfai_df = urbanfai_df[["月份", "当月", "自年初累计"]].rename(columns={"月份": "date", "当月": "current", "自年初累计": "cumulative"})

# Feb cumulative by year
year = urbanfai_df["date"].dt.year
month = urbanfai_df["date"].dt.month
feb_cum_by_year = urbanfai_df.loc[month.eq(2)].set_index(year[month.eq(2)])["cumulative"]

# fill Jan and Feb current = (Feb cumulative / 2)
for m in [1, 2]:
    mask = month.eq(m) & urbanfai_df["current"].isna()   # only fill missing
    urbanfai_df.loc[mask, "current"] = year[mask].map(feb_cum_by_year) / 2


In [22]:
proxyfai_df = fai_df.merge(urbanfai_df[["date", "current", "cumulative"]], on="date", how="outer").sort_values("date").reset_index(drop=True)

In [32]:

# ── Rebuild proxyfai_df with guaranteed proper DatetimeIndex ───────────────────
_fai = fai_df[['date', 'fai growth']].copy()
_fai['date'] = pd.to_datetime(_fai['date'], errors='coerce')

_urb = urbanfai_df[['date', 'current', 'cumulative']].copy()
_urb['date'] = pd.to_datetime(_urb['date'], errors='coerce')
_urb['cumulative'] = pd.to_numeric(_urb['cumulative'], errors='coerce')
_urb['current']    = pd.to_numeric(_urb['current'],    errors='coerce')

proxyfai_df = (
    _fai.merge(_urb, on='date', how='outer')
    .sort_values('date')
    .set_index('date')
)
proxyfai_df['fai growth'] = pd.to_numeric(proxyfai_df['fai growth'], errors='coerce')

print("Index sample:", proxyfai_df.index[:3].tolist())
print("Cumulative non-null:", proxyfai_df['cumulative'].notna().sum())

# ── Step 1: Find anchor year — earliest year >= 2008 with real cumulative data ─
valid_mask = (
    proxyfai_df['cumulative'].notna() &
    (proxyfai_df['cumulative'] > 0) &
    (proxyfai_df.index.year >= 2008)
)
if not valid_mask.any():
    raise ValueError("No valid cumulative data found in 2008 or later")
anchor_year = int(proxyfai_df.index[valid_mask].year.min())
print(f"Anchor year: {anchor_year}")

# ── Step 2: Backcast cumulative month-by-month from anchor_year back to 2000 ──
# Skip January (NBS doesn't report Jan separately); leave it NaN.
# Formula: cum[year, m] = cum[year+1, m] / (1 + fai_growth[year+1, m] / 100)
for year in range(anchor_year - 1, 1999, -1):
    for month in range(2, 13):
        date_curr = pd.Timestamp(year=year,     month=month, day=1)
        date_next = pd.Timestamp(year=year + 1, month=month, day=1)

        if date_curr not in proxyfai_df.index or date_next not in proxyfai_df.index:
            continue

        cum_next    = proxyfai_df.loc[date_next, 'cumulative']
        growth_next = proxyfai_df.loc[date_next, 'fai growth']

        if pd.notna(cum_next) and cum_next > 0 and pd.notna(growth_next):
            proxyfai_df.loc[date_curr, 'cumulative'] = cum_next / (1 + growth_next / 100)

# ── Step 3: Derive current values from cumulative ─────────────────────────────
for year in proxyfai_df.index.year.unique():
    for month in range(1, 13):
        date = pd.Timestamp(year=int(year), month=month, day=1)
        if date not in proxyfai_df.index:
            continue

        if month == 1:
            pass  # filled when Feb is processed below

        elif month == 2:
            feb_cum = proxyfai_df.loc[date, 'cumulative']
            if pd.notna(feb_cum):
                proxyfai_df.loc[date, 'current'] = feb_cum / 2
                jan = pd.Timestamp(year=int(year), month=1, day=1)
                if jan in proxyfai_df.index:
                    proxyfai_df.loc[jan, 'current'] = feb_cum / 2

        else:
            if pd.isna(proxyfai_df.loc[date, 'current']):
                prev = pd.Timestamp(year=int(year), month=month - 1, day=1)
                if prev in proxyfai_df.index:
                    cum_this = proxyfai_df.loc[date, 'cumulative']
                    cum_prev = proxyfai_df.loc[prev, 'cumulative']
                    if pd.notna(cum_this) and pd.notna(cum_prev):
                        proxyfai_df.loc[date, 'current'] = cum_this - cum_prev

print("\nResult (2000–2010):")
print(proxyfai_df.loc['2000':'2010', ['fai growth', 'cumulative', 'current']])


Index sample: [Timestamp('2000-01-01 00:00:00'), Timestamp('2000-02-01 00:00:00'), Timestamp('2000-03-01 00:00:00')]
Cumulative non-null: 199
Anchor year: 2008

Result (2000–2010):
            fai growth     cumulative       current
date                                               
2000-01-01         NaN            NaN    568.952440
2000-02-01         8.6    1137.904880    568.952440
2000-03-01         8.5    2528.888169   1390.983289
2000-04-01         9.3    4006.294087   1477.405917
2000-05-01         9.5    5886.383938   1880.089851
...                ...            ...           ...
2010-08-01        24.8  140997.740000  21131.500000
2010-09-01        24.5  165869.580000  24871.830000
2010-10-01        24.4  187556.100000  21686.530000
2010-11-01        24.9  210697.830000  23141.720000
2010-12-01        24.5  241414.930000  30717.100000

[132 rows x 3 columns]


In [33]:
proxyfai_df.columns = ["date", "proxy_fai", "proxy_cumulative_fai"]

## Government Spending

In [ ]:
gov_df = pd.read_csv(
    "NationalGovSpend.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)
gov_df = gov_df.iloc[[0], :].copy()

gov_df = gov_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
gov_df["date"] = pd.to_datetime(gov_df["date"], format="%Y年%m月", errors="coerce")

gov_df.columns = ["date", "gov_spend_ytd"]

In [39]:
# generate and calculate current values for gov spend
gov_df = gov_df.sort_values("date").reset_index(drop=True)
gov_df["gov_spend_ytd"] = pd.to_numeric(gov_df["gov_spend_ytd"], errors="coerce")

year  = gov_df["date"].dt.year
month = gov_df["date"].dt.month

# Feb cumulative by year
feb_cum_by_year = gov_df.loc[month.eq(2)].set_index(year[month.eq(2)])["gov_spend_ytd"]

gov_df["gov_spend_current"] = np.nan

# Jan and Feb: each = Feb cumulative / 2
for m in [1, 2]:
    mask = month.eq(m)
    gov_df.loc[mask, "gov_spend_current"] = year[mask].map(feb_cum_by_year) / 2

# Mar–Dec: current = cumulative_t - cumulative_{t-1}
# Use date-based lookup to avoid index-gap issues
ytd_by_date = gov_df.set_index("date")["gov_spend_ytd"]
mar_plus    = month.gt(2)
prev_dates  = gov_df.loc[mar_plus, "date"] - pd.DateOffset(months=1)
gov_df.loc[mar_plus, "gov_spend_current"] = (
    gov_df.loc[mar_plus, "gov_spend_ytd"].values
    - prev_dates.map(ytd_by_date).values
)

print(gov_df[["date", "gov_spend_ytd", "gov_spend_current"]])

          date  gov_spend_ytd  gov_spend_current
0   2000-01-01            NaN             748.85
1   2000-02-01         1497.7             748.85
2   2000-03-01         2376.4             878.70
3   2000-04-01         3463.8            1087.40
4   2000-05-01         4485.9            1022.10
..         ...            ...                ...
309 2025-10-01       225825.0           17761.00
310 2025-11-01       248538.0           22713.00
311 2025-12-01            NaN                NaN
312 2026-01-01            NaN                NaN
313 2026-02-01            NaN                NaN

[314 rows x 3 columns]


In [42]:
annualgov_df = pd.read_csv(
    "AnnualGovSpend.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)
annualgov_df = annualgov_df.iloc[[0], :].copy()

annualgov_df = annualgov_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
annualgov_df["date"] = pd.to_datetime(annualgov_df["date"], format="%Y年", errors="coerce")

annualgov_df.columns = ["date", "gov_spend_annual"]

In [43]:
# Fill missing December YTD and current values (2000–2009) from annual totals
annualgov_df["gov_spend_annual"] = pd.to_numeric(annualgov_df["gov_spend_annual"], errors="coerce")

# Build year → annual total lookup
annual_by_year = annualgov_df.set_index(annualgov_df["date"].dt.year)["gov_spend_annual"]

# Re-build date-based YTD lookup (needed to look up November after filling December)
ytd_by_date = gov_df.set_index("date")["gov_spend_ytd"]

dec_mask = (gov_df["date"].dt.month == 12) & (gov_df["date"].dt.year.between(2000, 2009))

for i in gov_df.index[dec_mask]:
    yr       = gov_df.loc[i, "date"].year
    annual   = annual_by_year.get(yr, np.nan)
    nov_date = pd.Timestamp(year=yr, month=11, day=1)
    nov_ytd  = ytd_by_date.get(nov_date, np.nan)

    if pd.notna(annual):
        gov_df.loc[i, "gov_spend_ytd"]     = annual
        gov_df.loc[i, "gov_spend_current"] = annual - nov_ytd if pd.notna(nov_ytd) else np.nan

print(gov_df.loc[gov_df["date"].dt.year.between(2000, 2010) & (gov_df["date"].dt.month.isin([10, 11, 12])),
                 ["date", "gov_spend_ytd", "gov_spend_current"]])

          date  gov_spend_ytd  gov_spend_current
9   2000-10-01       10533.70            1080.90
10  2000-11-01       11909.30            1375.60
11  2000-12-01       13395.23            1485.93
21  2001-10-01       12703.10            1223.50
22  2001-11-01       14504.50            1801.40
23  2001-12-01       16386.04            1881.54
33  2002-10-01       15011.40            1513.80
34  2002-11-01       16911.40            1900.00
35  2002-12-01       18903.64            1992.24
45  2003-10-01       16947.90            1683.00
46  2003-11-01       18932.00            1984.10
47  2003-12-01       21715.25            2783.25
57  2004-10-01       19061.90            1917.30
58  2004-11-01       21593.70            2531.80
59  2004-12-01       26396.47            4802.77
69  2005-10-01       22137.90            2188.00
70  2005-11-01       25326.30            3188.40
71  2005-12-01       31649.29            6322.99
81  2006-10-01       25827.50            2670.50
82  2006-11-01      

## CPI

In [47]:
cpi2015_df = pd.read_csv(
    "cpi2015.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)
cpi2015_df = cpi2015_df.iloc[[0], :].copy()

cpi2015_df = cpi2015_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
cpi2015_df["date"] = pd.to_datetime(cpi2015_df["date"], format="%Y年%m月", errors="coerce")

cpi2015_df.columns = ["date", "cpi"]

cpi2015_df["cpi"] = cpi2015_df["cpi"] - 100 
